# Genomic Selection Tutorial - Python Version

This notebook replicates the AlphaSimR genomic selection tutorial using AlphaSimPy.
It demonstrates genomic selection in a wheat line breeding program.

**Authors**: Translated from AlphaSimR tutorial by Jon Bancic, Philip Greenspoon, Chris Gaynor, Gregor Gorjanc  
**Date**: 2024  
**Package**: AlphaSimPy

This tutorial applies GS to advance individuals from DH to make PYT as well as to select new parents from DH stage.

## Import Required Libraries

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve
from AlphaSimPy import (
    runMacs, SimParam, newPop, randCross, setPheno, selectInd, 
    selectWithinFam, makeDH, meanG, varG, mergePops
)

print("AlphaSimPy Genomic Selection Tutorial")
print("All libraries imported successfully!")

AlphaSimPy Genomic Selection Tutorial
All libraries imported successfully!


## Helper Functions for Genomic Selection

These functions implement RRBLUP and setEBV functionality for genomic selection, plus a helper for subsetting Pop objects.

In [ ]:
def subsetPop(pop, indices):
    """
    Subset a Pop object by indices (helper function since Pop doesn't support indexing).
    
    Parameters:
    -----------
    pop : Pop
        Population object
    indices : list or slice
        Indices to select
    
    Returns:
    --------
    Pop
        Subsetted population
    """
    from AlphaSimPy import Pop
    
    if isinstance(indices, slice):
        indices = list(range(*indices.indices(pop.n_ind)))
    
    if not indices:
        # Return empty population
        return Pop(
            n_ind=0, n_chr=pop.n_chr, ploidy=pop.ploidy, n_loci=pop.n_loci,
            geno=[], gen_map=pop.gen_map, centromere=pop.centromere,
            inbred=pop.inbred, id=[], iid=[], mother=[], father=[],
            sex=[], n_traits=pop.n_traits, gv=np.empty((0, pop.n_traits)),
            pheno=np.empty((0, pop.n_traits)), ebv=np.empty((0, 0)),
            gxe=pop.gxe, fix_eff=[], misc={}, misc_pop={}
        )
    
    return Pop(
        n_ind=len(indices), n_chr=pop.n_chr, ploidy=pop.ploidy, n_loci=pop.n_loci,
        geno=[pop.geno[chr_idx][:, :, indices] for chr_idx in range(pop.n_chr)],
        gen_map=pop.gen_map, centromere=pop.centromere, inbred=pop.inbred,
        id=[pop.id[i] for i in indices],
        iid=[pop.iid[i] for i in indices],
        mother=[pop.mother[i] for i in indices],
        father=[pop.father[i] for i in indices],
        sex=[pop.sex[i] for i in indices],
        n_traits=pop.n_traits, gv=pop.gv[indices, :],
        pheno=pop.pheno[indices, :], ebv=pop.ebv[indices, :],
        gxe=pop.gxe, fix_eff=[pop.fix_eff[i] for i in indices],
        misc=pop.misc, misc_pop=pop.misc_pop
    )


def pullSnpGeno(pop, simParam, snpChip=1):
    """
    Extract SNP genotype matrix from a population.
    
    Parameters:
    -----------
    pop : Pop
        Population object
    simParam : SimParam
        Simulation parameters
    snpChip : int
        Which SNP chip to use (1-indexed)
    
    Returns:
    --------
    np.ndarray
        Matrix of SNP genotypes (n_ind x n_snp)
    """
    if snpChip < 1 or snpChip > len(simParam.snp_chips):
        raise ValueError(f"snpChip {snpChip} not available")
    
    snp_chip = simParam.snp_chips[snpChip - 1]
    n_snp = sum(snp_chip.loci_per_chr)
    
    if n_snp == 0:
        raise ValueError("No SNPs available in specified chip")
    
    # Extract genotypes for SNP loci
    geno_matrix = np.zeros((pop.n_ind, n_snp), dtype=np.float64)
    
    snp_idx = 0
    for chr_idx in range(pop.n_chr):
        n_snp_chr = snp_chip.loci_per_chr[chr_idx]
        if n_snp_chr == 0:
            continue
        
        # Get SNP locations on this chromosome
        snp_loci = snp_chip.loci_loc[snp_idx:snp_idx + n_snp_chr]
        
        # Extract genotypes from packed format
        for ind_idx in range(pop.n_ind):
            for snp_loc_idx, snp_loc in enumerate(snp_loci):
                # Extract genotype from packed byte format
                byte_idx = snp_loc // 8
                bit_idx = snp_loc % 8
                
                # Sum across ploidy
                genotype = 0
                for p in range(pop.ploidy):
                    byte_val = pop.geno[chr_idx][byte_idx, p, ind_idx]
                    bit_val = (byte_val >> bit_idx) & 1
                    genotype += bit_val
                
                geno_matrix[ind_idx, snp_idx + snp_loc_idx] = genotype
        
        snp_idx += n_snp_chr
    
    return geno_matrix


def RRBLUP(trainPop, simParam, traits=1, use="pheno", snpChip=1):
    """
    Fit an RR-BLUP model for genomic predictions.
    
    Parameters:
    -----------
    trainPop : Pop
        Training population
    simParam : SimParam
        Simulation parameters
    traits : int
        Trait to model (1-indexed)
    use : str
        Use "pheno", "gv", or "ebv" for training
    snpChip : int
        Which SNP chip to use
    
    Returns:
    --------
    dict
        Dictionary containing model coefficients and metadata
    """
    # Get response variable
    if use == "pheno":
        y = trainPop.pheno[:, traits - 1]
    elif use == "gv":
        y = trainPop.gv[:, traits - 1]
    elif use == "ebv":
        y = trainPop.ebv[:, traits - 1]
    else:
        raise ValueError(f"use='{use}' is not a valid option")
    
    # Remove missing values
    valid_idx = ~np.isnan(y)
    y = y[valid_idx]
    
    if len(y) == 0:
        raise ValueError("No valid observations for training")
    
    # Get SNP genotypes
    M = pullSnpGeno(trainPop, simParam, snpChip)
    M = M[valid_idx, :]
    
    # Center genotypes
    M_mean = np.mean(M, axis=0)
    M_centered = M - M_mean
    
    # Fit RR-BLUP using mixed model equations
    # y = Xb + Zu + e
    # where Z is the centered marker matrix
    # We use the GBLUP equivalent: K = ZZ'/p where p is number of markers
    
    n_markers = M_centered.shape[1]
    if n_markers == 0:
        raise ValueError("No markers available")
    
    # Calculate genomic relationship matrix G = ZZ' / p
    G = np.dot(M_centered, M_centered.T) / n_markers
    
    # Add small value to diagonal for numerical stability
    G += np.eye(G.shape[0]) * 1e-6
    
    # Estimate variance components (simplified - using fixed lambda)
    # In practice, you would estimate these, but for tutorial we use fixed values
    lambda_val = n_markers / 10.0  # Simplified lambda
    
    # Solve for BLUP: (G + lambda*I) * u = y
    # where u are the breeding values
    A = G + lambda_val * np.eye(G.shape[0])
    u = solve(A, y, assume_a='pos')
    
    # Store model for prediction
    model = {
        'u': u,  # BLUP solutions
        'M_mean': M_mean,  # Mean marker values for centering
        'M_train': M_centered,  # Training marker matrix (centered)
        'y_train': y,  # Training phenotypes
        'lambda': lambda_val,  # Regularization parameter
        'n_markers': n_markers,
        'trait': traits,
        'valid_idx': valid_idx
    }
    
    return model


def setEBV(pop, gsModel, simParam, snpChip=1):
    """
    Set estimated breeding values (EBV) for a population using a genomic selection model.
    
    Parameters:
    -----------
    pop : Pop
        Population to predict
    gsModel : dict
        Genomic selection model from RRBLUP
    simParam : SimParam
        Simulation parameters
    snpChip : int
        Which SNP chip to use
    
    Returns:
    --------
    Pop
        Population with EBV set
    """
    # Get SNP genotypes for prediction population
    M_pred = pullSnpGeno(pop, simParam, snpChip)
    
    # Center using training population means
    M_pred_centered = M_pred - gsModel['M_mean']
    
    # Calculate genomic relationship between training and prediction
    # G_pred_train = M_pred_centered @ M_train' / p
    n_markers = gsModel['n_markers']
    G_pred_train = np.dot(M_pred_centered, gsModel['M_train'].T) / n_markers
    
    # Predict EBV: u_pred = G_pred_train @ (G_train + lambda*I)^(-1) @ y
    # We already have u_train = (G_train + lambda*I)^(-1) @ y from RRBLUP
    # So: u_pred = G_pred_train @ u_train
    
    # Alternative: use the relationship matrix approach
    # Calculate G_pred (genomic relationship matrix for prediction pop)
    G_pred = np.dot(M_pred_centered, M_pred_centered.T) / n_markers
    
    # For prediction, we use: u_pred = G_pred_train @ solve(G_train + lambda*I, y)
    # Since we already solved for u_train, we can use:
    # u_pred = G_pred_train @ u_train
    
    # Calculate G_train for solving
    G_train = np.dot(gsModel['M_train'], gsModel['M_train'].T) / n_markers
    G_train += np.eye(G_train.shape[0]) * 1e-6
    
    # Solve: (G_train + lambda*I) * u = y
    A_train = G_train + gsModel['lambda'] * np.eye(G_train.shape[0])
    u_train = solve(A_train, gsModel['y_train'], assume_a='pos')
    
    # Predict: u_pred = G_pred_train @ u_train
    u_pred = np.dot(G_pred_train, u_train)
    
    # Set EBV in population
    if pop.ebv.shape[1] == 0:
        # Initialize EBV matrix if empty
        pop.ebv = np.zeros((pop.n_ind, 1))
    
    # Ensure EBV matrix has enough columns
    trait_idx = gsModel['trait'] - 1
    while pop.ebv.shape[1] <= trait_idx:
        pop.ebv = np.hstack([pop.ebv, np.zeros((pop.n_ind, 1))])
    
    pop.ebv[:, trait_idx] = u_pred
    
    return pop

## Global Parameters

Set up the simulation parameters for the genomic selection breeding program.

In [ ]:
# Number of simulation replications and breeding cycles
n_reps = 1  # Number of simulation replicates
n_burnin = 20  # Number of years in burnin phase
n_future = 20  # Number of years in future phase
n_cycles = n_burnin + n_future
start_tp = 19  # Year to start training population

# Genome simulation
n_chr = 10  # Number of chromosomes
n_qtl = 1000  # Number of QTL per chromosome
n_snp = 400  # Number of SNP per chromosome

# Initial parents mean and variance
init_mean_g = 1
init_var_g = 1
init_var_env = 1e-6  # Virtually zero for consistency with 2-Part paper
init_var_ge = 2
var_e = 4  # Yield trial error variance, bushels per acre
            # Relates to error variance for an entry mean

# Breeding program details
n_parents = 50  # Number of parents to start a breeding cycle
n_crosses = 100  # Number of crosses per year
n_dh = 100  # DH lines produced per cross
fam_max = 10  # The maximum number of DH lines per cross to enter PYT
n_pyt = 500  # Entries per preliminary yield trial
n_ayt = 50  # Entries per advanced yield trial
n_eyt = 10  # Entries per elite yield trial

# Effective replication of yield trials
rep_hdrw = 4/9  # h2 = 0.1
rep_pyt = 1  # h2 = 0.2
rep_ayt = 4  # h2 = 0.5
rep_eyt = 8  # h2 = 0.7

scenario_name = "LineGS"

print(f"Simulation Parameters:")
print(f"  Replicates: {n_reps}")
print(f"  Burn-in years: {n_burnin}")
print(f"  Future years: {n_future}")
print(f"  Total cycles: {n_cycles}")
print(f"  Start training population: Year {start_tp}")
print(f"  Chromosomes: {n_chr}")
print(f"  QTL per chromosome: {n_qtl}")
print(f"  SNP per chromosome: {n_snp}")

## Create Founders

Generate the initial founder population with haplotypes and set up simulation parameters.

In [ ]:
print("Creating founders...")

# Generate initial haplotypes
founder_pop = runMacs(nInd=n_parents, nChr=n_chr, segSites=n_qtl + n_snp,
                     inbred=True, species="WHEAT")

print(f"✓ Created founder population: {founder_pop.n_ind} individuals, {founder_pop.n_loci[0]} loci per chromosome")

# Create simulation parameters
SP = SimParam(founder_pop)
print(f"✓ Created SimParam: {SP.n_chr} chromosomes")

# Restrict segregating sites (separate QTL and SNP)
SP.restrSegSites(minQtlPerChr=n_qtl, minSnpPerChr=n_snp)

# Add SNP chip
if n_snp > 0:
    SP.addSnpChip(n_snp)
    print(f"✓ Added SNP chip: {SP.n_snp_chips} SNP chip(s)")

# Add traits: trait represents yield
SP.addTraitAG(nQtlPerChr=n_qtl, mean=init_mean_g, var=init_var_g, 
              varEnv=init_var_env, varGxE=init_var_ge)
print(f"✓ Added TraitAG: {SP.n_traits} traits")

# Collect pedigree
SP.setTrackPed(True)
print("✓ Enabled pedigree tracking")

# Create founder parents
Parents = newPop(founder_pop, simParam=SP)
print(f"✓ Created founder parents: {Parents.n_ind} individuals, {Parents.n_traits} traits")

# Add phenotype reflecting evaluation in EYT
Parents = setPheno(Parents, varE=var_e, reps=rep_eyt, simParam=SP)

print(f"\nFounder population summary:")
print(f"  Mean genetic value: {meanG(Parents)[0]:.3f}")
print(f"  Genetic variance: {varG(Parents)[0]:.3f}")

## Fill Breeding Pipeline

Set up the initial breeding pipeline with unique individuals from initial parents.

In [ ]:
print("Filling pipeline...")

# Set initial yield trials with unique individuals
for cohort in range(1, 8):
    print(f"  FillPipeline stage: {cohort} of 7")
    if cohort < 7:
        # Stage 1: Make crosses
        F1 = randCross(Parents, n_crosses, simParam=SP)
    if cohort < 6:
        # Stage 2: Make DH lines
        DH = makeDH(F1, n_dh, simParam=SP)
    if cohort < 5:
        # Stage 3: HDRW (High Density Row)
        HDRW = setPheno(DH, varE=var_e, reps=rep_hdrw, simParam=SP)
    if cohort < 4:
        # Stage 4: PYT (Preliminary Yield Trial)
        PYT = selectWithinFam(HDRW, fam_max, simParam=SP)
        PYT = selectInd(PYT, n_pyt, simParam=SP)
        PYT = setPheno(PYT, varE=var_e, reps=rep_pyt, simParam=SP)
    if cohort < 3:
        # Stage 5: AYT (Advanced Yield Trial)
        AYT = selectInd(PYT, n_ayt, simParam=SP)
        AYT = setPheno(AYT, varE=var_e, reps=rep_ayt, simParam=SP)
    if cohort < 2:
        # Stage 6: EYT (Elite Yield Trial)
        EYT = selectInd(AYT, n_eyt, simParam=SP)
        EYT = setPheno(EYT, varE=var_e, reps=rep_eyt, simParam=SP)
    if cohort < 1:
        # Stage 7: Release variety
        pass

print(f"\nPipeline filled successfully!")
print(f"  F1: {F1.n_ind} individuals")
print(f"  DH: {DH.n_ind} individuals")
print(f"  HDRW: {HDRW.n_ind} individuals")
print(f"  PYT: {PYT.n_ind} individuals")
print(f"  AYT: {AYT.n_ind} individuals")
print(f"  EYT: {EYT.n_ind} individuals")

## Main Simulation Loop

Run the breeding program with burn-in (phenotypic selection) and future (genomic selection) phases.

In [ ]:
# Create list to store results from reps
results = []

for REP in range(1, n_reps + 1):
    print(f"\n{'='*60}")
    print(f"Working on REP: {REP}")
    print(f"{'='*60}")

    # Create a data frame to track key parameters
    output = {
        'year': list(range(1, n_cycles + 1)),
        'rep': [REP] * n_cycles,
        'scenario': [scenario_name] * n_cycles,
        'meanG': [0.0] * n_cycles,
        'varG': [0.0] * n_cycles,
        'accSel': [0.0] * n_cycles
    }

    # Initialize training population
    TrainPop = None

    # ---- Burn-in phase: Phenotypic selection program ----
    print("\n--> Working on Burn-in")
    for year in range(1, n_burnin + 1):
        print(f" Working on burn-in year: {year}")
        
        # Update parents (phenotypic selection)
        # Replace 10 oldest inbred parents with 10 new parents from EYT stage
        if year > 1:
            Parents = mergePops([subsetPop(Parents, list(range(10, Parents.n_ind))), EYT])
        
        # Advance year (phenotypic selection)
        # Stage 6: EYT
        EYT = selectInd(AYT, n_eyt, simParam=SP)
        EYT = setPheno(EYT, varE=var_e, reps=rep_eyt, simParam=SP)
        
        # Stage 5: AYT
        AYT = selectInd(PYT, n_ayt, simParam=SP)
        AYT = setPheno(AYT, varE=var_e, reps=rep_ayt, simParam=SP)
        
        # Stage 4: PYT
        # Calculate selection accuracy
        acc_sel = np.corrcoef(HDRW.gv.flatten(), HDRW.pheno.flatten())[0, 1]
        output['accSel'][year - 1] = acc_sel if not np.isnan(acc_sel) else 0.0
        
        PYT = selectWithinFam(HDRW, fam_max, simParam=SP)
        PYT = selectInd(PYT, n_pyt, simParam=SP)
        PYT = setPheno(PYT, varE=var_e, reps=rep_pyt, simParam=SP)
        
        # Stage 3: HDRW
        HDRW = setPheno(DH, varE=var_e, reps=rep_hdrw, simParam=SP)
        
        # Stage 2: DH
        DH = makeDH(F1, n_dh, simParam=SP)
        
        # Stage 1: F1
        F1 = randCross(Parents, n_crosses, simParam=SP)
        
        # Store training population
        if year == start_tp:
            print("  Start collecting training population")
            TrainPop = mergePops([PYT, EYT, AYT])
        elif year > start_tp and year < n_burnin + 1:
            print("  Collecting training population")
            TrainPop = mergePops([TrainPop, PYT, EYT, AYT])
        
        # Report results
        output['meanG'][year - 1] = meanG(DH)[0]
        output['varG'][year - 1] = varG(DH)[0]

    # ---- Future phase: Genomic selection program ----
    print("\n--> Working on Genomic line breeding program")
    for year in range(n_burnin + 1, n_burnin + n_future + 1):
        print(f" Working on future year: {year}")
        
        # Run genomic model
        print("  Running GS model")
        gsModel = RRBLUP(TrainPop, SP, traits=1, use="pheno", snpChip=1)
        
        # Update parents (genomic selection)
        # Replace 10 oldest inbred parents with 10 new genomically-predicted inbreds from DH stage
        
        # Predict EBV of DHs
        DH = setEBV(DH, gsModel, SP, snpChip=1)
        
        # Select 10 new parents based on EBVs
        newParents = selectInd(DH, 10, use="ebv", simParam=SP)
        
        # Replace 10 oldest inbred parents with 10 new inbreds from DH stage
        Parents = mergePops([subsetPop(Parents, list(range(10, Parents.n_ind))), newParents])
        
        # Advance year (genomic selection)
        # Stage 6: EYT
        EYT = selectInd(AYT, n_eyt, simParam=SP)
        EYT = setPheno(EYT, varE=var_e, reps=rep_eyt, simParam=SP)
        
        # Stage 5: AYT
        AYT = selectInd(PYT, n_ayt, simParam=SP)
        AYT = setPheno(AYT, varE=var_e, reps=rep_ayt, simParam=SP)
        
        # Stage 4: PYT (using genomic selection)
        # NOTE: HDRW removed because phenotyping not needed
        DH = setEBV(DH, gsModel, SP, snpChip=1)
        
        # Calculate selection accuracy
        acc_sel = np.corrcoef(DH.gv.flatten(), DH.ebv.flatten())[0, 1]
        output['accSel'][year - 1] = acc_sel if not np.isnan(acc_sel) else 0.0
        
        PYT = selectWithinFam(DH, fam_max, use="ebv", simParam=SP)
        PYT = selectInd(PYT, n_pyt, use="ebv", simParam=SP)
        PYT = setPheno(PYT, varE=var_e, reps=rep_pyt, simParam=SP)
        
        # Stage 2: DH
        DH = makeDH(F1, n_dh, simParam=SP)
        
        # Stage 1: F1
        F1 = randCross(Parents, n_crosses, simParam=SP)
        
        # Store training population (maintain by removing oldest, adding newest)
        print("  Maintaining training population")
        # Remove oldest entries (size of PYT + EYT + AYT)
        n_remove = PYT.n_ind + EYT.n_ind + AYT.n_ind
        if TrainPop.n_ind > n_remove:
            TrainPop = mergePops([subsetPop(TrainPop, list(range(n_remove, TrainPop.n_ind))), PYT, EYT, AYT])
        else:
            TrainPop = mergePops([PYT, EYT, AYT])
        
        # Report results
        output['meanG'][year - 1] = meanG(DH)[0]
        output['varG'][year - 1] = varG(DH)[0]

    # Save results from current replicate
    results.append(output)

print("\n" + "="*60)
print("Simulation completed!")
print("="*60)

## Analyze Results

Plot the results showing genetic gain, genetic variance, and selection accuracy over time.

In [ ]:
# Combine results from all replicates
if len(results) > 0:
    # Extract data
    years = np.array(results[0]['year'])
    mean_g_all = np.array([r['meanG'] for r in results])
    var_g_all = np.array([r['varG'] for r in results])
    acc_sel_all = np.array([r['accSel'] for r in results])
    
    # Calculate means across replicates
    mean_g_avg = np.mean(mean_g_all, axis=0)
    var_g_avg = np.mean(var_g_all, axis=0)
    acc_sel_avg = np.mean(acc_sel_all, axis=0)
    
    # Create plots
    fig, axes = plt.subplots(3, 1, figsize=(6, 12))
    
    # Genetic Gain
    axes[0].plot(years, mean_g_avg, 'b-', linewidth=2)
    axes[0].axvline(x=n_burnin, color='r', linestyle='--', label='GS Start')
    axes[0].set_title('Genetic gain', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Year', fontsize=12)
    axes[0].set_ylabel('Yield', fontsize=12)
    axes[0].grid(True, linestyle=':', alpha=0.7)
    axes[0].legend()
    
    # Genetic Variance
    axes[1].plot(years, var_g_avg, 'b-', linewidth=2)
    axes[1].axvline(x=n_burnin, color='r', linestyle='--', label='GS Start')
    axes[1].set_title('Genetic variance', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Year', fontsize=12)
    axes[1].set_ylabel('Variance', fontsize=12)
    axes[1].grid(True, linestyle=':', alpha=0.7)
    axes[1].legend()
    
    # Selection Accuracy
    axes[2].plot(years, acc_sel_avg, 'b-', linewidth=2)
    axes[2].axvline(x=n_burnin, color='r', linestyle='--', label='GS Start')
    axes[2].set_title('Selection accuracy', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Year', fontsize=12)
    axes[2].set_ylabel('Correlation', fontsize=12)
    axes[2].grid(True, linestyle=':', alpha=0.7)
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig('GenomicSelection_Results.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print(f"  Initial mean G: {mean_g_avg[0]:.3f}")
    print(f"  Final mean G: {mean_g_avg[-1]:.3f}")
    print(f"  Genetic gain: {mean_g_avg[-1] - mean_g_avg[0]:.3f}")
    print(f"  Initial variance: {var_g_avg[0]:.3f}")
    print(f"  Final variance: {var_g_avg[-1]:.3f}")
    print(f"  Average selection accuracy (burn-in): {np.mean(acc_sel_avg[:n_burnin]):.3f}")
    print(f"  Average selection accuracy (GS): {np.mean(acc_sel_avg[n_burnin:]):.3f}")
else:
    print("No results to plot.")